In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from llama_cpp import Llama
from llama_cpp.llama_speculative import LlamaPromptLookupDecoding, LlamaDraftModel
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from langchain.chat_models import init_chat_model
from openai import OpenAI


In [3]:
OPENAI_API_KEY = "YOUR_API_KEY"
import os
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY


llm = init_chat_model("gpt-4o-mini", model_provider="openai")

In [4]:
model_path='quants/hermes-3-2-3B.gguf'

In [21]:
model = Llama(
    model_path,
    n_gpu_layers=-1,
    n_ctx=4096,
    temperature=0.1,
    # draft_model=LlamaPromptLookupDecoding(num_pred_tokens=10),
)

ggml_cuda_init: GGML_CUDA_FORCE_MMQ:    yes
ggml_cuda_init: GGML_CUDA_FORCE_CUBLAS: no
ggml_cuda_init: found 1 CUDA devices:
  Device 0: NVIDIA GeForce RTX 4090, compute capability 8.9, VMM: yes
llama_load_model_from_file: using device CUDA0 (NVIDIA GeForce RTX 4090) - 6031 MiB free
llama_model_loader: loaded meta data with 38 key-value pairs and 255 tensors from quants/hermes-3-2-3B.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Hermes 3 Llama 3.2 3b Base Fft Chatml...
llama_model_loader: - kv   3:                            general.version str              = 2024-11-24
llama_model_loader: - kv   4:                      

In [6]:
from constrerl.annotator import Annotator, Article, load_train, load_test

In [22]:
data_path = "data/annotations/dev/dev.json"
out_path = "data/results/dev_out.json"
# annotator = Annotator(langchain=llm, gen_tokens=2048)
annotator = Annotator(model=model, gen_tokens=2048)
eval_set = load_train(data_path)
annotator.add_prompt_examples([a for a in eval_set.values()][0:1])

In [23]:
with open("grammar.gbnf", "w") as f:
    f.write(annotator.erl_grammar)
print(annotator.erl_grammar)

root ::= (" "| "\n") grammar-models
grammar-models ::= relations
relations ::= "{"  ws "\"relations\"" ": " relations-relations  ws "}"
relation ::= "{"  ws "\"link_type\"" ": " relation-link-type ","  ws "\"subject_text_span\"" ": " string ","  ws "\"subject_location\"" ": " relation-subject-location ","  ws "\"object_text_span\"" ": " string ","  ws "\"object_location\"" ": " relation-object-location  ws "}"
relation-link-type ::= "\"anatomical location | located in | human\"" | "\"anatomical location | located in | animal\"" | "\"bacteria | interact | bacteria\"" | "\"bacteria | interact | chemical\"" | "\"bacteria | interact | drug\"" | "\"bacteria | influence | DDF\"" | "\"bacteria | change expression | gene\"" | "\"bacteria | located in | human\"" | "\"bacteria | located in | animal\"" | "\"bacteria | part of | microbiome\"" | "\"chemical | located in | anatomical location\"" | "\"chemical | located in | human\"" | "\"chemical | located in | animal\"" | "\"chemical | interact | c

In [24]:
annotator.example_messages

[{'role': 'system',
  'content': 'You are annotating a medical scientific title and abstract. You return only the most relevant relations within the title and abstract as JSON. The few returned relations include the relation type and text. It is very important to return only the ten most relevant relations.\n'},
 {'role': 'user',
  'content': "Title: Hypothesis of a potential BrainBiota and its relation to CNS autoimmune inflammation.\nAbstract: Infectious agents have been long considered to play a role in the pathogenesis of neurological diseases as part of the interaction between genetic susceptibility and the environment. The role of bacteria in CNS autoimmunity has also been highlighted by changes in the diversity of gut microbiota in patients with neurological diseases such as Parkinson's disease, Alzheimer disease and multiple sclerosis, emphasizing the role of the gut-brain axis. We discuss the hypothesis of a brain microbiota, the BrainBiota: bacteria living in symbiosis with b

In [ ]:
annotations = annotator.annotate(
    {id: article.metadata for id, article in list(eval_set.items())[2:10]}
)

Annotating articles:   0%|          | 0/8 [00:00<?, ?it/s]

llama_perf_context_print:        load time =     239.47 ms
llama_perf_context_print: prompt eval time =       0.00 ms /  1054 tokens (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:        eval time =       0.00 ms /  2047 runs   (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:       total time =   69942.53 ms /  3101 tokens
Annotating articles:  12%|█▎        | 1/8 [01:09<08:09, 69.95s/it, id=37577447]Llama.generate: 747 prefix-match hit, remaining 375 prompt tokens to eval


Error in article 37577447
 {
  "relations": [
    {
      "link_type": "microbiome | compared to | microbiome",
      "subject_text_span": "oral microbiota",
      "subject_location": "abstract",
      "object_text_span": "gut microbiota",
      "object_location": "abstract"
    },
    {
      "link_type": "microbiome | located in | human",
      "subject_text_span": "gut microbiota",
      "subject_location": "abstract",
      "object_text_span": "patients",
      "object_location": "abstract"
    },
    {
      "link_type": "microbiome | located in | human",
      "subject_text_span": "oral microbiota",
      "subject_location": "abstract",
      "object_text_span": "patients",
      "object_location": "abstract"
    },
    {
      "link_type": "microbiome | compared to | microbiome",
      "subject_text_span": "persons with AD",
      "subject_location": "abstract",
      "object_text_span": "healthy controls",
      "object_location": "abstract"
    },
    {
      "link_type": "mic

llama_perf_context_print:        load time =     239.47 ms
llama_perf_context_print: prompt eval time =       0.00 ms /   375 tokens (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:        eval time =       0.00 ms /  2047 runs   (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:       total time =   70924.30 ms /  2422 tokens
Annotating articles:  25%|██▌       | 2/8 [02:20<07:03, 70.53s/it, id=38203207]Llama.generate: 747 prefix-match hit, remaining 383 prompt tokens to eval


Error in article 38203207
 {
  "relations": [
    {
      "link_type": "microbiome | is linked to | DDF",
      "subject_text_span": "gut microbiota",
      "subject_location": "abstract",
      "object_text_span": "neuropsychiatric disorders",
      "object_location": "abstract"
    },
    {
      "link_type": "microbiome | is linked to | DDF",
      "subject_text_span": "gut microbiome",
      "subject_location": "abstract",
      "object_text_span": "autism",
      "object_location": "abstract"
    },
    {
      "link_type": "microbiome | is linked to | DDF",
      "subject_text_span": "gut microbiome",
      "subject_location": "abstract",
      "object_text_span": "depression",
      "object_location": "abstract"
    },
    {
      "link_type": "microbiome | is linked to | DDF",
      "subject_text_span": "gut microbiome",
      "subject_location": "abstract",
      "object_text_span": "schizophrenia",
      "object_location": "abstract"
    },
    {
      "link_type": "microbiom

Annotating articles:  25%|██▌       | 2/8 [02:23<07:11, 71.97s/it, id=38203207]


KeyboardInterrupt: 

: 

In [18]:
annotations

{'37577447': Relations(relations=[Relation(link_type=<LinkType.microbiome_located_in_anatomical_location: 'microbiome | located in | anatomical location'>, subject_text_span='BrainBiota', subject_location=<LabelLocation.TITLE: 'title'>, object_text_span='human brain', object_location=<LabelLocation.ABSTRACT: 'abstract'>), Relation(link_type=<LinkType.microbiome_is_linked_to_DDF: 'microbiome | is linked to | DDF'>, subject_text_span='gut microbiota', subject_location=<LabelLocation.ABSTRACT: 'abstract'>, object_text_span='CNS autoimmunity', object_location=<LabelLocation.ABSTRACT: 'abstract'>), Relation(link_type=<LinkType.bacteria_interact_chemical: 'bacteria | interact | chemical'>, subject_text_span='bacteria', subject_location=<LabelLocation.ABSTRACT: 'abstract'>, object_text_span='bacterial proteins', object_location=<LabelLocation.ABSTRACT: 'abstract'>), Relation(link_type=<LinkType.bacteria_influence_DDF: 'bacteria | influence | DDF'>, subject_text_span='bacteria', subject_locati

In [19]:
import numpy as np
np.mean(np.array([len(a.relations) for a in annotations.values()]))

np.float64(10.0)